# 21. Neural Network Architecture Exploration

This notebook explores why the NN predictions for different edge types from the executed notebooks 04_model_testing follow a tight distribution, but do not accurately predict the empirical edge frequencies. As a sanity check, single-layer NN architectures are compared to logistic regression since a single layer NN should align perfectly with the logistic regression model. Then, it tests whether the prediction function is appropriate. 

## Key Questions to Address:
1. Why are NN predictions tight but not diagonal? 
2. Can a single-layer NN match logistic regression (sanity check)?
3. Would alternative architectures (GNN, CNN, RNN) learn edge probabilities better?
4. For longer paths: when do edges transition from conditional to independent?
5. Can RNNs model degree signature sequences for metapaths?

# 1. Import Libraries, Load Data, and Prepare Features

In [ ]:
import os
import sys
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
import scipy.sparse as sp
from scipy.stats import pearsonr
from typing import Dict, List, Tuple, Optional
import warnings

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Add src to path - try both relative paths
sys.path.insert(0, '../src')
sys.path.insert(0, 'src')

# Import project modules
from model_comparison import prepare_edge_features_and_labels, ModelCollection, filter_zero_degree_nodes, SimpleNN
from model_evaluation import ModelEvaluator, evaluate_model
from simple_models import SingleLayerNN

# Create results directory if it doesn't exist
os.makedirs('../results/', exist_ok=True)

# Define results directory
results_dir = '../results/nn_optimizer_comparison'
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Setup paths (matching notebook 04 methodology)
repo_dir = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
data_dir = repo_dir / 'data'

# Select an edge type for testing (matching notebook 4 parameter structure)
edge_type = 'CbG'  # Compound-binds-Gene
edge_file_path = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'{edge_type}.sparse.npz'

print(f"Testing with edge type: {edge_type}")
print(f"Loading from: {edge_file_path}")
print(f"File exists: {edge_file_path.exists()}")

# Validate edge file exists
if not edge_file_path.exists():
    raise FileNotFoundError(
        f"Edge file not found: {edge_file_path}\n"
        f"Please ensure the data directory structure is correct."
    )

# Apply zero-degree filtering first (following notebook 4's approach)
print("Loading edge matrix for zero-degree filtering...")
edge_matrix = sp.load_npz(str(edge_file_path))
print(f"Original edge matrix: {edge_matrix.shape} with {edge_matrix.nnz} edges")

# Apply zero-degree node filtering
print("\nApplying zero-degree filtering...")
filtered_edge_matrix, source_mapping, target_mapping = filter_zero_degree_nodes(edge_matrix)
print(f"Filtered edge matrix: {filtered_edge_matrix.shape} with {filtered_edge_matrix.nnz} edges")
print(f"Removed {edge_matrix.shape[0] - filtered_edge_matrix.shape[0]} source nodes and {edge_matrix.shape[1] - filtered_edge_matrix.shape[1]} target nodes with degree=0")
print(f"Density improvement: {edge_matrix.nnz / (edge_matrix.shape[0] * edge_matrix.shape[1]):.6f} → {filtered_edge_matrix.nnz / (filtered_edge_matrix.shape[0] * filtered_edge_matrix.shape[1]):.6f}")

# Save filtered matrix temporarily for feature creation
filtered_edge_path = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'filtered_{edge_type}.sparse.npz'
sp.save_npz(str(filtered_edge_path), filtered_edge_matrix)

# Load features and labels with intelligent adaptive sampling (notebook 4's method)
print("\nPreparing edge features and labels from filtered data...")
try:
    X, y = prepare_edge_features_and_labels(
        str(filtered_edge_path),
        sample_ratio=0.01,  # Matching notebook 4's approach
        adaptive_sampling=True,
        enhanced_features=False  # Use only basic 2 features
    )
    
    # Clean up temporary file
    filtered_edge_path.unlink()
    
    # Basic validation
    if X.shape[0] == 0:
        raise ValueError("No samples loaded from edge file")
    if X.shape[1] != 2:
        raise ValueError(f"Expected 2 features (source_degree, target_degree), got {X.shape[1]}")
        
    print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
    print(f"Positive samples: {y.sum()}, Negative samples: {(1-y).sum()}")
    print(f"Class balance: {y.sum()/len(y):.3f}")
    
except Exception as e:
    raise RuntimeError(f"Failed to prepare features and labels: {e}")

Testing with edge type: CbG
Loading edge matrix for zero-degree filtering...
Original edge matrix: (1552, 20945) with 11571 edges

Applying zero-degree filtering...

  Zero-degree node filtering:
    Removed 163 sources and 19256 targets with degree=0
    Retained: 1389 sources × 1689 targets
    Density: 0.000356 → 0.004932 (13.9x increase)
Filtered edge matrix: (1389, 1689) with 11571 edges
Removed 163 source nodes and 19256 target nodes with degree=0
Density improvement: 0.000356 → 0.004932

Preparing edge features and labels from filtered data...
Original edge matrix:
  Shape: (1389, 1689)
  Edges: 11,571

Filtered edge matrix statistics:
  Shape: (1389, 1689)
  Existing edges: 11,571
  Edge density: 0.004932 (0.493%)
  Mean degrees: source=8.33, target=6.85
  Adaptive sampling analysis:
    Size score: 0.813, Sparsity score: 0.049, Degree score: 0.759
    Combined score: 0.497 → Adapted ratio: 0.080
    This gives ~143784 total samples (11571 pos + 132213 neg)
  Sampling ratio: 0.

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train positive ratio: {y_train.sum()/len(y_train):.3f}")
print(f"Test positive ratio: {y_test.sum()/len(y_test):.3f}")


In [ ]:
# Calculate pos_weight for class balancing (for Tests 3 and 4)
num_negatives = (y_train == 0).sum()
num_positives = (y_train == 1).sum()
pos_weight = torch.tensor([num_negatives / num_positives])

print(f"Class balance in training set:")
print(f"  Positive examples: {num_positives}")
print(f"  Negative examples: {num_negatives}")
print(f"  pos_weight: {pos_weight.item():.2f}")


In [ ]:
# Create PyTorch tensors for training

X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train)
y_test_tensor = torch.FloatTensor(y_test)

print("\nCreated PyTorch tensors:")
print(f"  X_train_tensor: {X_train_tensor.shape}")
print(f"  X_test_tensor: {X_test_tensor.shape}")
print(f"  y_train_tensor: {y_train_tensor.shape}")
print(f"  y_test_tensor: {y_test_tensor.shape}")


## Test 1: Single Layer NN (Adam) vs Logistic Regression

This test compares:
1. Single-layer neural network trained with Adam optimizer
2. Logistic regression (exact configuration from notebook 04)

Results are saved for reuse in Test 2.


In [ ]:
# SingleLayerNN model is imported from src/simple_models.py
# See Cell 2 for import statement
print("SingleLayerNN imported from src/simple_models.py")

In [ ]:
# Train single-layer NN with Adam optimizer
print("\n" + "="*60)
print("Training Single-Layer NN with Adam Optimizer")
print("="*60)

model_adam = SingleLayerNN()
optimizer_adam = torch.optim.Adam(model_adam.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# Add OneCycleLR scheduler (matching notebook 04)
scheduler_adam = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_adam,
    max_lr=0.01,              # 10x base learning rate
    epochs=500,
    steps_per_epoch=1,        # Full-batch training
    pct_start=0.3,            # Warm-up for first 30%
    anneal_strategy='cos'     # Cosine annealing
)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_adam = []
test_losses_adam = []
best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
converged_epoch_adam = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    # Training
    model_adam.train()
    optimizer_adam.zero_grad()
    outputs = model_adam(X_train_tensor)
    loss = criterion(outputs.squeeze(), y_train_tensor)
    loss.backward()
    optimizer_adam.step()
    scheduler_adam.step()  # Step scheduler each epoch
    train_losses_adam.append(loss.item())
    
    # Testing
    model_adam.eval()
    with torch.no_grad():
        test_outputs = model_adam(X_test_tensor)
        test_loss = criterion(test_outputs.squeeze(), y_test_tensor)
        test_losses_adam.append(test_loss.item())
    
    # Early stopping check
    if test_loss.item() < best_loss - min_delta:
        best_loss = test_loss.item()
        best_model_state = model_adam.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        current_lr = scheduler_adam.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}, LR: {current_lr:.6f}")
    
    # Check for convergence
    if epochs_no_improve >= patience:
        converged_epoch_adam = epoch + 1
        print(f"\nEarly stopping at epoch {converged_epoch_adam}")
        # Restore best model
        model_adam.load_state_dict(best_model_state)
        break

train_time_adam = time.time() - start_time
print(f"\nTraining completed in {train_time_adam:.2f} seconds")

# Get predictions
model_adam.eval()
with torch.no_grad():
    train_pred_adam = torch.sigmoid(model_adam(X_train_tensor)).numpy().flatten()
    test_pred_adam = torch.sigmoid(model_adam(X_test_tensor)).numpy().flatten()

print(f"Predictions shape - Train: {train_pred_adam.shape}, Test: {test_pred_adam.shape}")


In [ ]:
# Train logistic regression (exact config from notebook 04)
print("\n" + "="*60)
print("Training Logistic Regression (Notebook 04 Configuration)")
print("="*60)
logreg = LogisticRegression(
    random_state=42,
    max_iter=1000
    # Uses sklearn defaults: solver='lbfgs', penalty='l2', C=1.0
)
start_time = time.time()
logreg.fit(X_train, y_train)
train_time_logreg = time.time() - start_time
# Get predictions (probabilities)
train_pred_logreg = logreg.predict_proba(X_train)[:, 1]
test_pred_logreg = logreg.predict_proba(X_test)[:, 1]
print(f"Training completed in {train_time_logreg:.2f} seconds")
print(f"Predictions shape - Train: {train_pred_logreg.shape}, Test: {test_pred_logreg.shape}")


In [ ]:
# evaluate_model function is imported from src/model_evaluation.py
# See Cell 2 for import statement

# Evaluate on test set
metrics_adam = evaluate_model(y_test, test_pred_adam, 'Single Layer NN (Adam)')
metrics_logreg = evaluate_model(y_test, test_pred_logreg, 'Logistic Regression')

# Print results
print("\n" + "="*60)
print("Test 1 Evaluation Results")
print("="*60)
for metrics in [metrics_adam, metrics_logreg]:
    print(f"\n{metrics['model']}:")
    print(f"  AUC: {metrics['auc']:.4f}")
    print(f"  Average Precision: {metrics['average_precision']:.4f}")

In [ ]:
# Save Test 1 results
# Save Single Layer NN (Adam) results
adam_results = {
    'model': model_adam,
    'train_pred': train_pred_adam,
    'test_pred': test_pred_adam,
    'train_losses': train_losses_adam,
    'test_losses': test_losses_adam,
    'metrics': metrics_adam,
    'train_time': train_time_adam,
    'num_epochs': max_epochs
}
with open(f'{results_dir}/single_layer_nn_adam.pkl', 'wb') as f:
    pickle.dump(adam_results, f)
print(f"Saved: {results_dir}/single_layer_nn_adam.pkl")
# Save Logistic Regression results
logreg_results = {
    'model': logreg,
    'train_pred': train_pred_logreg,
    'test_pred': test_pred_logreg,
    'metrics': metrics_logreg,
    'train_time': train_time_logreg
}
with open(f'{results_dir}/logistic_regression.pkl', 'wb') as f:
    pickle.dump(logreg_results, f)
print(f"Saved: {results_dir}/logistic_regression.pkl")
print("\nTest 1 results saved successfully!")


In [ ]:
# Visualize Test 1 loss curves
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(train_losses_adam, label='Train Loss', linewidth=2)
ax.plot(test_losses_adam, label='Test Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12)
ax.set_title('Single Layer NN (Adam) - Training History', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/single_layer_nn_adam_loss.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {results_dir}/single_layer_nn_adam_loss.png")


## Test 2: Add L-BFGS Optimizer Comparison

This test:
1. Loads results from Test 1 (Adam NN and Logistic Regression)
2. Trains single-layer NN with L-BFGS optimizer
3. Compares all three approaches

No redundant calculations are performed.


In [ ]:
# Load Test 1 results
print("Loading Test 1 results...")
with open(f'{results_dir}/single_layer_nn_adam.pkl', 'rb') as f:
    adam_results = pickle.load(f)
with open(f'{results_dir}/logistic_regression.pkl', 'rb') as f:
    logreg_results = pickle.load(f)
print("Test 1 results loaded successfully!")
print(f"  Adam NN - Test AUC: {adam_results['metrics']['auc']:.4f}")
print(f"  LogReg - Test AUC: {logreg_results['metrics']['auc']:.4f}")


In [ ]:
# Train single-layer NN with L-BFGS optimizer
print("\n" + "="*60)
print("Training Single-Layer NN with L-BFGS Optimizer")
print("="*60)

model_lbfgs = SingleLayerNN()
optimizer_lbfgs = torch.optim.LBFGS(
    model_lbfgs.parameters(),
    lr=0.01,
    max_iter=20
)
criterion = nn.BCEWithLogitsLoss()

# Note: L-BFGS has built-in adaptive step sizing, no external scheduler needed

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_lbfgs = []
test_losses_lbfgs = []
best_loss = float('inf')
epochs_no_improve = 0
converged_epoch = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_lbfgs.train()
    
    def closure():
        optimizer_lbfgs.zero_grad()
        outputs = model_lbfgs(X_train_tensor)
        loss = criterion(outputs.squeeze(), y_train_tensor)
        loss.backward()
        return loss
    
    loss = optimizer_lbfgs.step(closure)
    train_losses_lbfgs.append(loss.item())
    
    # Testing
    model_lbfgs.eval()
    with torch.no_grad():
        test_outputs = model_lbfgs(X_test_tensor)
        test_loss = criterion(test_outputs.squeeze(), y_test_tensor)
        test_losses_lbfgs.append(test_loss.item())
    
    # Check convergence
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch = epoch + 1
        print(f"\nConverged at epoch {converged_epoch}")
        break

train_time_lbfgs = time.time() - start_time
print(f"\nTraining completed in {train_time_lbfgs:.2f} seconds")

# Get predictions
model_lbfgs.eval()
with torch.no_grad():
    train_pred_lbfgs = torch.sigmoid(model_lbfgs(X_train_tensor)).numpy().flatten()
    test_pred_lbfgs = torch.sigmoid(model_lbfgs(X_test_tensor)).numpy().flatten()

print(f"Predictions shape - Train: {train_pred_lbfgs.shape}, Test: {test_pred_lbfgs.shape}")


In [ ]:
# Evaluate all three models

metrics_lbfgs = evaluate_model(y_test, test_pred_lbfgs, 'Single Layer NN (L-BFGS)')

# Print comparison
print("\n" + "="*60)
print("Test 2 - Complete Model Comparison")
print("="*60)

all_metrics = [
    adam_results['metrics'],
    metrics_lbfgs,
    logreg_results['metrics']
]

for metrics in all_metrics:
    print(f"\n{metrics['model']}:")
    print(f"  AUC: {metrics['auc']:.4f}")
    print(f"  Average Precision: {metrics['average_precision']:.4f}")

# Create comparison dataframe
comparison_df = pd.DataFrame(all_metrics)
print("\nComparison Table:")
print(comparison_df.to_string(index=False))


In [ ]:
# Save Test 2 results

# Save L-BFGS results
lbfgs_results = {
    'model': model_lbfgs,
    'train_pred': train_pred_lbfgs,
    'test_pred': test_pred_lbfgs,
    'train_losses': train_losses_lbfgs,
    'test_losses': test_losses_lbfgs,
    'metrics': metrics_lbfgs,
    'train_time': train_time_lbfgs,
    'converged_epoch': converged_epoch
}

with open(f'{results_dir}/single_layer_nn_lbfgs.pkl', 'wb') as f:
    pickle.dump(lbfgs_results, f)
print(f"Saved: {results_dir}/single_layer_nn_lbfgs.pkl")

# Save combined comparison
comparison_results = {
    'models': ['Single Layer NN (Adam)', 'Single Layer NN (L-BFGS)', 'Logistic Regression'],
    'metrics_df': comparison_df,
    'test_predictions': {
        'adam': adam_results['test_pred'],
        'lbfgs': test_pred_lbfgs,
        'logreg': logreg_results['test_pred']
    },
    'true_labels': y_test
}

with open(f'{results_dir}/comparison_summary.pkl', 'wb') as f:
    pickle.dump(comparison_results, f)
print(f"Saved: {results_dir}/comparison_summary.pkl")

# Save comparison table as CSV
comparison_df.to_csv(f'{results_dir}/model_comparison.csv', index=False)
print(f"Saved: {results_dir}/model_comparison.csv")

print("\nTest 2 results saved successfully!")


In [ ]:
# Visualize L-BFGS loss curves

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(train_losses_lbfgs, label='Train Loss', linewidth=2)
ax.plot(test_losses_lbfgs, label='Test Loss', linewidth=2)
ax.axvline(x=converged_epoch-1, color='red', linestyle='--', 
           label=f'Converged (Epoch {converged_epoch})', alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12)
ax.set_title('Single Layer NN (L-BFGS) - Training History', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/single_layer_nn_lbfgs_loss.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/single_layer_nn_lbfgs_loss.png")


In [ ]:
# Create combined comparison plot

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Adam optimizer
axes[0].plot(adam_results['train_losses'], label='Train Loss', linewidth=2)
axes[0].plot(adam_results['test_losses'], label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12)
axes[0].set_title('Single Layer NN (Adam)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# L-BFGS optimizer
axes[1].plot(train_losses_lbfgs, label='Train Loss', linewidth=2)
axes[1].plot(test_losses_lbfgs, label='Test Loss', linewidth=2)
axes[1].axvline(x=converged_epoch-1, color='red', linestyle='--', 
                label=f'Converged (Epoch {converged_epoch})', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12)
axes[1].set_title('Single Layer NN (L-BFGS)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Optimizer Comparison - Training Histories', 
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(f'{results_dir}/optimizer_comparison_losses.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/optimizer_comparison_losses.png")


In [ ]:
# Create Precision-Recall curves comparing all models

from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

models_data = [
    ('Single Layer NN (Adam)', adam_results['test_pred'], adam_results['metrics']),
    ('Single Layer NN (L-BFGS)', test_pred_lbfgs, metrics_lbfgs),
    ('Logistic Regression', logreg_results['test_pred'], logreg_results['metrics'])
]

line_styles = ['-', '--', ':']  # solid, dashed, dotted
colors = ['C0', 'C1', 'C2']

for (name, preds, metrics), style, color in zip(models_data, line_styles, colors):
    precision, recall, _ = precision_recall_curve(y_test, preds)
    ax.plot(recall, precision, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AP={metrics['average_precision']:.3f})")

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/precision_recall_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/precision_recall_comparison.png")


In [ ]:
# Create ROC curves comparing all models

from sklearn.metrics import roc_curve

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

line_styles = ['-', '--', ':']  # solid, dashed, dotted
colors = ['C0', 'C1', 'C2']

for (name, preds, metrics), style, color in zip(models_data, line_styles, colors):
    fpr, tpr, _ = roc_curve(y_test, preds)
    ax.plot(fpr, tpr, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AUC={metrics['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')

# Add note about identical performance
auc_diff = abs(metrics_lbfgs['auc'] - logreg_results['metrics']['auc'])
ax.text(0.6, 0.2, f"L-BFGS & LogReg\ndiffer by {auc_diff:.5f}",
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/roc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/roc_comparison.png")


## Test 3: Class Weighting Comparison + SimpleNN

This test:
1. Adds class weighting (`pos_weight`) to all single-layer models
2. Adds SimpleNN from notebook 04 for comparison
3. Compares all 4 models: Adam NN, L-BFGS NN, LogReg, SimpleNN

All models use the existing `pos_weight` calculated in the data loading cell.

In [ ]:
# Train single-layer NN (Adam) with class weighting
print("\n" + "="*60)
print("Test 3: Single-Layer NN (Adam) with Class Weighting")
print("="*60)

model_adam_weighted = SingleLayerNN()
optimizer_adam_weighted = torch.optim.Adam(model_adam_weighted.parameters(), lr=0.001)
criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Add OneCycleLR scheduler
scheduler_adam_weighted = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_adam_weighted,
    max_lr=0.01,
    epochs=500,
    steps_per_epoch=1,
    pct_start=0.3,
    anneal_strategy='cos'
)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_adam_weighted = []
test_losses_adam_weighted = []
best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
converged_epoch_adam_weighted = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    # Training
    model_adam_weighted.train()
    optimizer_adam_weighted.zero_grad()
    outputs = model_adam_weighted(X_train_tensor)
    loss = criterion_weighted(outputs.squeeze(), y_train_tensor)
    loss.backward()
    optimizer_adam_weighted.step()
    scheduler_adam_weighted.step()
    train_losses_adam_weighted.append(loss.item())
    
    # Testing
    model_adam_weighted.eval()
    with torch.no_grad():
        test_outputs = model_adam_weighted(X_test_tensor)
        test_loss = criterion_weighted(test_outputs.squeeze(), y_test_tensor)
        test_losses_adam_weighted.append(test_loss.item())
    
    # Early stopping check
    if test_loss.item() < best_loss - min_delta:
        best_loss = test_loss.item()
        best_model_state = model_adam_weighted.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        current_lr = scheduler_adam_weighted.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}, LR: {current_lr:.6f}")
    
    # Check for convergence
    if epochs_no_improve >= patience:
        converged_epoch_adam_weighted = epoch + 1
        print(f"\nEarly stopping at epoch {converged_epoch_adam_weighted}")
        model_adam_weighted.load_state_dict(best_model_state)
        break

train_time_adam_weighted = time.time() - start_time
print(f"\nTraining completed in {train_time_adam_weighted:.2f} seconds")

# Get predictions
model_adam_weighted.eval()
with torch.no_grad():
    test_pred_adam_weighted = torch.sigmoid(model_adam_weighted(X_test_tensor)).numpy().flatten()

# Evaluate
metrics_adam_weighted = evaluate_model(y_test, test_pred_adam_weighted, 'Single Layer NN (Adam, Weighted)')
print(f"\nTest AUC: {metrics_adam_weighted['auc']:.4f}")
print(f"Test AP: {metrics_adam_weighted['average_precision']:.4f}")


In [ ]:
# Train single-layer NN (L-BFGS) with class weighting
print("\n" + "="*60)
print("Test 3: Single-Layer NN (L-BFGS) with Class Weighting")
print("="*60)

model_lbfgs_weighted = SingleLayerNN()
optimizer_lbfgs_weighted = torch.optim.LBFGS(
    model_lbfgs_weighted.parameters(),
    lr=0.01,
    max_iter=20
)
criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Note: L-BFGS has built-in adaptive step sizing, no external scheduler needed

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_lbfgs_weighted = []
test_losses_lbfgs_weighted = []
best_loss = float('inf')
epochs_no_improve = 0
converged_epoch_lbfgs_weighted = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_lbfgs_weighted.train()
    
    def closure():
        optimizer_lbfgs_weighted.zero_grad()
        outputs = model_lbfgs_weighted(X_train_tensor)
        loss = criterion_weighted(outputs.squeeze(), y_train_tensor)
        loss.backward()
        return loss
    
    loss = optimizer_lbfgs_weighted.step(closure)
    train_losses_lbfgs_weighted.append(loss.item())
    
    # Testing
    model_lbfgs_weighted.eval()
    with torch.no_grad():
        test_outputs = model_lbfgs_weighted(X_test_tensor)
        test_loss = criterion_weighted(test_outputs.squeeze(), y_test_tensor)
        test_losses_lbfgs_weighted.append(test_loss.item())
    
    # Check convergence
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_lbfgs_weighted = epoch + 1
        print(f"\nConverged at epoch {converged_epoch_lbfgs_weighted}")
        break

train_time_lbfgs_weighted = time.time() - start_time
print(f"\nTraining completed in {train_time_lbfgs_weighted:.2f} seconds")

# Get predictions
model_lbfgs_weighted.eval()
with torch.no_grad():
    test_pred_lbfgs_weighted = torch.sigmoid(model_lbfgs_weighted(X_test_tensor)).numpy().flatten()

# Evaluate
metrics_lbfgs_weighted = evaluate_model(y_test, test_pred_lbfgs_weighted, 'Single Layer NN (L-BFGS, Weighted)')
print(f"\nTest AUC: {metrics_lbfgs_weighted['auc']:.4f}")
print(f"Test AP: {metrics_lbfgs_weighted['average_precision']:.4f}")


In [ ]:
# Train Logistic Regression with class weighting
print("\n" + "="*60)
print("Test 3: Logistic Regression with Class Weighting")
print("="*60)

logreg_weighted = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

start_time = time.time()
logreg_weighted.fit(X_train, y_train)
train_time_logreg_weighted = time.time() - start_time

# Get predictions
test_pred_logreg_weighted = logreg_weighted.predict_proba(X_test)[:, 1]

# Evaluate
metrics_logreg_weighted = evaluate_model(y_test, test_pred_logreg_weighted, 'Logistic Regression (Weighted)')

print(f"Training completed in {train_time_logreg_weighted:.2f} seconds")
print(f"\nTest AUC: {metrics_logreg_weighted['auc']:.4f}")
print(f"Test AP: {metrics_logreg_weighted['average_precision']:.4f}")


In [ ]:
# Train SimpleNN from Notebook 04
print("\n" + "="*60)
print("Test 3: SimpleNN from Notebook 04")
print("="*60)

# Instantiate SimpleNN
simple_nn = SimpleNN(
    input_dim=2,
    hidden_dims=(128, 64, 32),
    dropout_rate=0.3,
    use_class_weights=True
)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)

# Setup training
criterion_simplenn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_simplenn = optim.AdamW(
    simple_nn.parameters(),
    lr=0.001,
    weight_decay=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8
)

# OneCycleLR scheduler
scheduler_simplenn = optim.lr_scheduler.OneCycleLR(
    optimizer_simplenn,
    max_lr=0.01,
    epochs=60,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

# Training loop
num_epochs_simplenn = 60
patience_simplenn = 5
best_loss_simplenn = float('inf')
epochs_no_improve_simplenn = 0
best_model_state_simplenn = None

train_losses_simplenn = []
test_losses_simplenn = []

start_time = time.time()

for epoch in range(num_epochs_simplenn):
    # Training
    simple_nn.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer_simplenn.zero_grad()
        outputs = simple_nn(batch_X)
        loss = criterion_simplenn(outputs.squeeze(), batch_y)
        loss.backward()
        optimizer_simplenn.step()
        scheduler_simplenn.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses_simplenn.append(avg_train_loss)
    
    # Testing
    simple_nn.eval()
    with torch.no_grad():
        test_outputs = simple_nn(X_test_tensor)
        test_loss = criterion_simplenn(test_outputs.squeeze(), y_test_tensor)
        test_losses_simplenn.append(test_loss.item())
    
    # Early stopping
    if test_loss.item() < best_loss_simplenn:
        best_loss_simplenn = test_loss.item()
        best_model_state_simplenn = simple_nn.state_dict().copy()
        epochs_no_improve_simplenn = 0
    else:
        epochs_no_improve_simplenn += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_simplenn} - "
              f"Train Loss: {avg_train_loss:.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve_simplenn >= patience_simplenn:
        print(f"\nEarly stopping at epoch {epoch+1}")
        simple_nn.load_state_dict(best_model_state_simplenn)
        break

train_time_simplenn = time.time() - start_time
print(f"\nTraining completed in {train_time_simplenn:.2f} seconds")

# Get predictions
simple_nn.eval()
with torch.no_grad():
    test_pred_simplenn = torch.sigmoid(simple_nn(X_test_tensor)).numpy().flatten()

# Evaluate
metrics_simplenn = evaluate_model(y_test, test_pred_simplenn, 'SimpleNN (Notebook 04)')
print(f"\nTest AUC: {metrics_simplenn['auc']:.4f}")
print(f"Test AP: {metrics_simplenn['average_precision']:.4f}")


In [ ]:
# Evaluate all Test 3 models
print("\n" + "="*60)
print("Test 3 - Complete Model Comparison")
print("="*60)

test3_metrics = [
    metrics_adam_weighted,
    metrics_lbfgs_weighted,
    metrics_logreg_weighted,
    metrics_simplenn
]

for metrics in test3_metrics:
    print(f"\n{metrics['model']}:")
    print(f"  AUC: {metrics['auc']:.4f}")
    print(f"  Average Precision: {metrics['average_precision']:.4f}")

# Create comparison dataframe
test3_comparison_df = pd.DataFrame(test3_metrics)
print("\nTest 3 Comparison Table:")
print(test3_comparison_df.to_string(index=False))

# Compare with Test 1 and Test 2
print("\n" + "="*60)
print("Comparison Across All Tests")
print("="*60)
print("\nTest 1 & 2 (No Class Weighting):")
print(f"  Adam NN:     AUC={adam_results['metrics']['auc']:.4f}")
print(f"  L-BFGS NN:   AUC={metrics_lbfgs['auc']:.4f}")
print(f"  LogReg:      AUC={logreg_results['metrics']['auc']:.4f}")
print("\nTest 3 (With Class Weighting):")
print(f"  Adam NN:     AUC={metrics_adam_weighted['auc']:.4f}")
print(f"  L-BFGS NN:   AUC={metrics_lbfgs_weighted['auc']:.4f}")
print(f"  LogReg:      AUC={metrics_logreg_weighted['auc']:.4f}")
print(f"  SimpleNN:    AUC={metrics_simplenn['auc']:.4f}")


In [ ]:
# Create Precision-Recall curves for Test 3
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

test3_models_data = [
    ('Adam NN (Weighted)', test_pred_adam_weighted, metrics_adam_weighted),
    ('L-BFGS NN (Weighted)', test_pred_lbfgs_weighted, metrics_lbfgs_weighted),
    ('LogReg (Weighted)', test_pred_logreg_weighted, metrics_logreg_weighted),
    ('SimpleNN', test_pred_simplenn, metrics_simplenn)
]

line_styles = ['-', '--', '-.', ':']
colors = ['C0', 'C1', 'C2', 'C3']

for (name, preds, metrics), style, color in zip(test3_models_data, line_styles, colors):
    precision, recall, _ = precision_recall_curve(y_test, preds)
    ax.plot(recall, precision, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AP={metrics['average_precision']:.3f})")

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Test 3: Precision-Recall Curves with Class Weighting', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/test3_precision_recall.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/test3_precision_recall.png")


In [ ]:
# Create ROC curves for Test 3
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

line_styles = ['-', '--', '-.', ':']
colors = ['C0', 'C1', 'C2', 'C3']

for (name, preds, metrics), style, color in zip(test3_models_data, line_styles, colors):
    fpr, tpr, _ = roc_curve(y_test, preds)
    ax.plot(fpr, tpr, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AUC={metrics['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Test 3: ROC Curves with Class Weighting', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/test3_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/test3_roc_curves.png")


In [ ]:
# Save Test 3 results

# Save Adam NN (weighted) results
adam_weighted_results = {
    'model': model_adam_weighted,
    'test_pred': test_pred_adam_weighted,
    'train_losses': train_losses_adam_weighted,
    'test_losses': test_losses_adam_weighted,
    'metrics': metrics_adam_weighted,
    'train_time': train_time_adam_weighted,
    'converged_epoch': converged_epoch_adam_weighted
}
with open(f'{results_dir}/test3_single_layer_nn_adam_weighted.pkl', 'wb') as f:
    pickle.dump(adam_weighted_results, f)
print(f"Saved: {results_dir}/test3_single_layer_nn_adam_weighted.pkl")

# Save L-BFGS NN (weighted) results
lbfgs_weighted_results = {
    'model': model_lbfgs_weighted,
    'test_pred': test_pred_lbfgs_weighted,
    'train_losses': train_losses_lbfgs_weighted,
    'test_losses': test_losses_lbfgs_weighted,
    'metrics': metrics_lbfgs_weighted,
    'train_time': train_time_lbfgs_weighted,
    'converged_epoch': converged_epoch_lbfgs_weighted
}
with open(f'{results_dir}/test3_single_layer_nn_lbfgs_weighted.pkl', 'wb') as f:
    pickle.dump(lbfgs_weighted_results, f)
print(f"Saved: {results_dir}/test3_single_layer_nn_lbfgs_weighted.pkl")

# Save LogReg (weighted) results
logreg_weighted_results = {
    'model': logreg_weighted,
    'test_pred': test_pred_logreg_weighted,
    'metrics': metrics_logreg_weighted,
    'train_time': train_time_logreg_weighted
}
with open(f'{results_dir}/test3_logreg_weighted.pkl', 'wb') as f:
    pickle.dump(logreg_weighted_results, f)
print(f"Saved: {results_dir}/test3_logreg_weighted.pkl")

# Save SimpleNN results
simplenn_results = {
    'model': simple_nn,
    'test_pred': test_pred_simplenn,
    'train_losses': train_losses_simplenn,
    'test_losses': test_losses_simplenn,
    'metrics': metrics_simplenn,
    'train_time': train_time_simplenn
}
with open(f'{results_dir}/test3_simple_nn.pkl', 'wb') as f:
    pickle.dump(simplenn_results, f)
print(f"Saved: {results_dir}/test3_simple_nn.pkl")

# Save comparison CSV
test3_comparison_df.to_csv(f'{results_dir}/test3_model_comparison.csv', index=False)
print(f"Saved: {results_dir}/test3_model_comparison.csv")

print("\nTest 3 results saved successfully!")


## Test 4: StandardScaler Transformation + Class Weighting

This test:
1. Applies StandardScaler to normalize features (matching notebook 04)
2. Trains all models with class weighting on scaled data
3. Provides direct comparison with notebook 04's methodology

Expected: Scaled features should improve convergence speed for all models.

In [ ]:
# Apply StandardScaler transformation (matching notebook 04)
from sklearn.preprocessing import StandardScaler

print("\n" + "="*60)
print("Test 4: Applying StandardScaler Transformation")
print("="*60)

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_scaled_tensor = torch.FloatTensor(X_train_scaled)
X_test_scaled_tensor = torch.FloatTensor(X_test_scaled)

print("\nFeature statistics before scaling:")
print(f"  Source degrees: {X_train[:, 0].min():.2f} - {X_train[:, 0].max():.2f} (mean: {X_train[:, 0].mean():.2f})")
print(f"  Target degrees: {X_train[:, 1].min():.2f} - {X_train[:, 1].max():.2f} (mean: {X_train[:, 1].mean():.2f})")

print("\nFeature statistics after scaling:")
print(f"  Source degrees: {X_train_scaled[:, 0].min():.2f} - {X_train_scaled[:, 0].max():.2f} (mean: {X_train_scaled[:, 0].mean():.2f}, std: {X_train_scaled[:, 0].std():.2f})")
print(f"  Target degrees: {X_train_scaled[:, 1].min():.2f} - {X_train_scaled[:, 1].max():.2f} (mean: {X_train_scaled[:, 1].mean():.2f}, std: {X_train_scaled[:, 1].std():.2f})")

print("\nScaled tensors created:")
print(f"  X_train_scaled_tensor: {X_train_scaled_tensor.shape}")
print(f"  X_test_scaled_tensor: {X_test_scaled_tensor.shape}")


In [ ]:
# Train single-layer NN (Adam) with scaled data and class weighting
print("\n" + "="*60)
print("Test 4: Single-Layer NN (Adam, Weighted, Scaled)")
print("="*60)

model_adam_scaled = SingleLayerNN()
optimizer_adam_scaled = torch.optim.Adam(model_adam_scaled.parameters(), lr=0.001)
criterion_scaled = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

scheduler_adam_scaled = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_adam_scaled,
    max_lr=0.01,
    epochs=500,
    steps_per_epoch=1,
    pct_start=0.3,
    anneal_strategy='cos'
)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_adam_scaled = []
test_losses_adam_scaled = []
best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
converged_epoch_adam_scaled = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_adam_scaled.train()
    optimizer_adam_scaled.zero_grad()
    outputs = model_adam_scaled(X_train_scaled_tensor)
    loss = criterion_scaled(outputs.squeeze(), y_train_tensor)
    loss.backward()
    optimizer_adam_scaled.step()
    scheduler_adam_scaled.step()
    train_losses_adam_scaled.append(loss.item())
    
    model_adam_scaled.eval()
    with torch.no_grad():
        test_outputs = model_adam_scaled(X_test_scaled_tensor)
        test_loss = criterion_scaled(test_outputs.squeeze(), y_test_tensor)
        test_losses_adam_scaled.append(test_loss.item())
    
    if test_loss.item() < best_loss - min_delta:
        best_loss = test_loss.item()
        best_model_state = model_adam_scaled.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        current_lr = scheduler_adam_scaled.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}, LR: {current_lr:.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_adam_scaled = epoch + 1
        print(f"\nEarly stopping at epoch {converged_epoch_adam_scaled}")
        model_adam_scaled.load_state_dict(best_model_state)
        break

train_time_adam_scaled = time.time() - start_time
print(f"\nTraining completed in {train_time_adam_scaled:.2f} seconds")

model_adam_scaled.eval()
with torch.no_grad():
    test_pred_adam_scaled = torch.sigmoid(model_adam_scaled(X_test_scaled_tensor)).numpy().flatten()

metrics_adam_scaled = evaluate_model(y_test, test_pred_adam_scaled, 'Single Layer NN (Adam, Weighted, Scaled)')
print(f"\nTest AUC: {metrics_adam_scaled['auc']:.4f}")
print(f"Test AP: {metrics_adam_scaled['average_precision']:.4f}")


In [ ]:
# Train single-layer NN (L-BFGS) with scaled data and class weighting
print("\n" + "="*60)
print("Test 4: Single-Layer NN (L-BFGS, Weighted, Scaled)")
print("="*60)

model_lbfgs_scaled = SingleLayerNN()
optimizer_lbfgs_scaled = torch.optim.LBFGS(
    model_lbfgs_scaled.parameters(),
    lr=0.01,
    max_iter=20
)
criterion_scaled = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Note: L-BFGS has built-in adaptive step sizing, no external scheduler needed

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_lbfgs_scaled = []
test_losses_lbfgs_scaled = []
best_loss = float('inf')
epochs_no_improve = 0
converged_epoch_lbfgs_scaled = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_lbfgs_scaled.train()
    
    def closure():
        optimizer_lbfgs_scaled.zero_grad()
        outputs = model_lbfgs_scaled(X_train_scaled_tensor)
        loss = criterion_scaled(outputs.squeeze(), y_train_tensor)
        loss.backward()
        return loss
    
    loss = optimizer_lbfgs_scaled.step(closure)
    train_losses_lbfgs_scaled.append(loss.item())
    
    model_lbfgs_scaled.eval()
    with torch.no_grad():
        test_outputs = model_lbfgs_scaled(X_test_scaled_tensor)
        test_loss = criterion_scaled(test_outputs.squeeze(), y_test_tensor)
        test_losses_lbfgs_scaled.append(test_loss.item())
    
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_lbfgs_scaled = epoch + 1
        print(f"\nConverged at epoch {converged_epoch_lbfgs_scaled}")
        break

train_time_lbfgs_scaled = time.time() - start_time
print(f"\nTraining completed in {train_time_lbfgs_scaled:.2f} seconds")

model_lbfgs_scaled.eval()
with torch.no_grad():
    test_pred_lbfgs_scaled = torch.sigmoid(model_lbfgs_scaled(X_test_scaled_tensor)).numpy().flatten()

metrics_lbfgs_scaled = evaluate_model(y_test, test_pred_lbfgs_scaled, 'Single Layer NN (L-BFGS, Weighted, Scaled)')
print(f"\nTest AUC: {metrics_lbfgs_scaled['auc']:.4f}")
print(f"Test AP: {metrics_lbfgs_scaled['average_precision']:.4f}")


In [ ]:
# Train Logistic Regression with scaled data and class weighting
print("\n" + "="*60)
print("Test 4: Logistic Regression (Weighted, Scaled)")
print("="*60)

logreg_scaled = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

start_time = time.time()
logreg_scaled.fit(X_train_scaled, y_train)
train_time_logreg_scaled = time.time() - start_time

test_pred_logreg_scaled = logreg_scaled.predict_proba(X_test_scaled)[:, 1]

metrics_logreg_scaled = evaluate_model(y_test, test_pred_logreg_scaled, 'Logistic Regression (Weighted, Scaled)')

print(f"Training completed in {train_time_logreg_scaled:.2f} seconds")
print(f"\nTest AUC: {metrics_logreg_scaled['auc']:.4f}")
print(f"Test AP: {metrics_logreg_scaled['average_precision']:.4f}")


In [ ]:
# Train SimpleNN with scaled data and class weighting (matching notebook 04)
print("\n" + "="*60)
print("Test 4: SimpleNN (Weighted, Scaled) - Notebook 04 Configuration")
print("="*60)

simple_nn_scaled = SimpleNN(
    input_dim=2,
    hidden_dims=(128, 64, 32),
    dropout_rate=0.3,
    use_class_weights=True
)

train_dataset_scaled = TensorDataset(X_train_scaled_tensor, y_train_tensor)
train_loader_scaled = DataLoader(train_dataset_scaled, batch_size=4096, shuffle=True)

criterion_simplenn_scaled = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_simplenn_scaled = optim.AdamW(
    simple_nn_scaled.parameters(),
    lr=0.001,
    weight_decay=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler_simplenn_scaled = optim.lr_scheduler.OneCycleLR(
    optimizer_simplenn_scaled,
    max_lr=0.01,
    epochs=60,
    steps_per_epoch=len(train_loader_scaled),
    pct_start=0.3,
    anneal_strategy='cos'
)

num_epochs_simplenn = 60
patience_simplenn = 5
best_loss_simplenn = float('inf')
epochs_no_improve_simplenn = 0
best_model_state_simplenn = None

train_losses_simplenn_scaled = []
test_losses_simplenn_scaled = []

start_time = time.time()

for epoch in range(num_epochs_simplenn):
    simple_nn_scaled.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader_scaled:
        optimizer_simplenn_scaled.zero_grad()
        outputs = simple_nn_scaled(batch_X)
        loss = criterion_simplenn_scaled(outputs.squeeze(), batch_y)
        loss.backward()
        optimizer_simplenn_scaled.step()
        scheduler_simplenn_scaled.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader_scaled)
    train_losses_simplenn_scaled.append(avg_train_loss)
    
    simple_nn_scaled.eval()
    with torch.no_grad():
        test_outputs = simple_nn_scaled(X_test_scaled_tensor)
        test_loss = criterion_simplenn_scaled(test_outputs.squeeze(), y_test_tensor)
        test_losses_simplenn_scaled.append(test_loss.item())
    
    if test_loss.item() < best_loss_simplenn:
        best_loss_simplenn = test_loss.item()
        best_model_state_simplenn = simple_nn_scaled.state_dict().copy()
        epochs_no_improve_simplenn = 0
    else:
        epochs_no_improve_simplenn += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_simplenn} - "
              f"Train Loss: {avg_train_loss:.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve_simplenn >= patience_simplenn:
        print(f"\nEarly stopping at epoch {epoch+1}")
        simple_nn_scaled.load_state_dict(best_model_state_simplenn)
        break

train_time_simplenn_scaled = time.time() - start_time
print(f"\nTraining completed in {train_time_simplenn_scaled:.2f} seconds")

simple_nn_scaled.eval()
with torch.no_grad():
    test_pred_simplenn_scaled = torch.sigmoid(simple_nn_scaled(X_test_scaled_tensor)).numpy().flatten()

metrics_simplenn_scaled = evaluate_model(y_test, test_pred_simplenn_scaled, 'SimpleNN (Weighted, Scaled)')
print(f"\nTest AUC: {metrics_simplenn_scaled['auc']:.4f}")
print(f"Test AP: {metrics_simplenn_scaled['average_precision']:.4f}")


In [ ]:
# Evaluate all Test 4 models
print("\n" + "="*60)
print("Test 4 - Complete Model Comparison")
print("="*60)

test4_metrics = [
    metrics_adam_scaled,
    metrics_lbfgs_scaled,
    metrics_logreg_scaled,
    metrics_simplenn_scaled
]

for metrics in test4_metrics:
    print(f"\n{metrics['model']}:")
    print(f"  AUC: {metrics['auc']:.4f}")
    print(f"  Average Precision: {metrics['average_precision']:.4f}")

test4_comparison_df = pd.DataFrame(test4_metrics)
print("\nTest 4 Comparison Table:")
print(test4_comparison_df.to_string(index=False))

# Compare across all tests
print("\n" + "="*60)
print("Comparison Across All Tests")
print("="*60)
print("\nTest 1 & 2 (No Class Weighting, No Scaling):")
print(f"  Adam NN:     AUC={adam_results['metrics']['auc']:.4f}")
print(f"  L-BFGS NN:   AUC={metrics_lbfgs['auc']:.4f}")
print(f"  LogReg:      AUC={logreg_results['metrics']['auc']:.4f}")
print("\nTest 3 (Class Weighting, No Scaling):")
print(f"  Adam NN:     AUC={metrics_adam_weighted['auc']:.4f}")
print(f"  L-BFGS NN:   AUC={metrics_lbfgs_weighted['auc']:.4f}")
print(f"  LogReg:      AUC={metrics_logreg_weighted['auc']:.4f}")
print(f"  SimpleNN:    AUC={metrics_simplenn['auc']:.4f}")
print("\nTest 4 (Class Weighting + Scaling):")
print(f"  Adam NN:     AUC={metrics_adam_scaled['auc']:.4f}")
print(f"  L-BFGS NN:   AUC={metrics_lbfgs_scaled['auc']:.4f}")
print(f"  LogReg:      AUC={metrics_logreg_scaled['auc']:.4f}")
print(f"  SimpleNN:    AUC={metrics_simplenn_scaled['auc']:.4f}")


In [ ]:
# Create Precision-Recall curves for Test 4
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

test4_models_data = [
    ('Adam NN (Scaled)', test_pred_adam_scaled, metrics_adam_scaled),
    ('L-BFGS NN (Scaled)', test_pred_lbfgs_scaled, metrics_lbfgs_scaled),
    ('LogReg (Scaled)', test_pred_logreg_scaled, metrics_logreg_scaled),
    ('SimpleNN (Scaled)', test_pred_simplenn_scaled, metrics_simplenn_scaled)
]

line_styles = ['-', '--', '-.', ':']
colors = ['C0', 'C1', 'C2', 'C3']

for (name, preds, metrics), style, color in zip(test4_models_data, line_styles, colors):
    precision, recall, _ = precision_recall_curve(y_test, preds)
    ax.plot(recall, precision, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AP={metrics['average_precision']:.3f})")

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Test 4: Precision-Recall Curves (Scaled Features)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/test4_precision_recall.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/test4_precision_recall.png")


In [ ]:
# Create ROC curves for Test 4
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

line_styles = ['-', '--', '-.', ':']
colors = ['C0', 'C1', 'C2', 'C3']

for (name, preds, metrics), style, color in zip(test4_models_data, line_styles, colors):
    fpr, tpr, _ = roc_curve(y_test, preds)
    ax.plot(fpr, tpr, linestyle=style, linewidth=2.5, color=color,
            label=f"{name} (AUC={metrics['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Test 4: ROC Curves (Scaled Features)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f'{results_dir}/test4_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/test4_roc_curves.png")


In [ ]:
# Save Test 4 results

# Save Adam NN (weighted, scaled) results
adam_scaled_results = {
    'model': model_adam_scaled,
    'test_pred': test_pred_adam_scaled,
    'train_losses': train_losses_adam_scaled,
    'test_losses': test_losses_adam_scaled,
    'metrics': metrics_adam_scaled,
    'train_time': train_time_adam_scaled,
    'converged_epoch': converged_epoch_adam_scaled,
    'scaler': scaler
}
with open(f'{results_dir}/test4_single_layer_nn_adam_weighted_scaled.pkl', 'wb') as f:
    pickle.dump(adam_scaled_results, f)
print(f"Saved: {results_dir}/test4_single_layer_nn_adam_weighted_scaled.pkl")

# Save L-BFGS NN (weighted, scaled) results
lbfgs_scaled_results = {
    'model': model_lbfgs_scaled,
    'test_pred': test_pred_lbfgs_scaled,
    'train_losses': train_losses_lbfgs_scaled,
    'test_losses': test_losses_lbfgs_scaled,
    'metrics': metrics_lbfgs_scaled,
    'train_time': train_time_lbfgs_scaled,
    'converged_epoch': converged_epoch_lbfgs_scaled,
    'scaler': scaler
}
with open(f'{results_dir}/test4_single_layer_nn_lbfgs_weighted_scaled.pkl', 'wb') as f:
    pickle.dump(lbfgs_scaled_results, f)
print(f"Saved: {results_dir}/test4_single_layer_nn_lbfgs_weighted_scaled.pkl")

# Save LogReg (weighted, scaled) results
logreg_scaled_results = {
    'model': logreg_scaled,
    'test_pred': test_pred_logreg_scaled,
    'metrics': metrics_logreg_scaled,
    'train_time': train_time_logreg_scaled,
    'scaler': scaler
}
with open(f'{results_dir}/test4_logreg_weighted_scaled.pkl', 'wb') as f:
    pickle.dump(logreg_scaled_results, f)
print(f"Saved: {results_dir}/test4_logreg_weighted_scaled.pkl")

# Save SimpleNN (weighted, scaled) results
simplenn_scaled_results = {
    'model': simple_nn_scaled,
    'test_pred': test_pred_simplenn_scaled,
    'train_losses': train_losses_simplenn_scaled,
    'test_losses': test_losses_simplenn_scaled,
    'metrics': metrics_simplenn_scaled,
    'train_time': train_time_simplenn_scaled,
    'scaler': scaler
}
with open(f'{results_dir}/test4_simple_nn_weighted_scaled.pkl', 'wb') as f:
    pickle.dump(simplenn_scaled_results, f)
print(f"Saved: {results_dir}/test4_simple_nn_weighted_scaled.pkl")

# Save comparison CSV
test4_comparison_df.to_csv(f'{results_dir}/test4_model_comparison.csv', index=False)
print(f"Saved: {results_dir}/test4_model_comparison.csv")

print("\nTest 4 results saved successfully!")
print("\nAll tests complete! Summary:")
print("  Test 1: Baseline comparison (no class weighting)")
print("  Test 2: Added L-BFGS optimizer")
print("  Test 3: Added class weighting to all models")
print("  Test 4: Added StandardScaler transformation (matching notebook 04)")


## Test 5: MSE Loss (No Class Weighting)

This test replaces BCEWithLogitsLoss with MSELoss to address sigmoid compression:
1. Neural networks trained with MSELoss instead of BCEWithLogitsLoss
2. Ridge regression (natural MSE-based model)
3. No class weighting
4. No StandardScaler

Expected: Better probability calibration, higher Pearson correlation with empirical frequencies

In [ ]:
# Import Ridge regression for Test 5 and Test 6
from sklearn.linear_model import Ridge

print("Ridge regression imported for MSE-based training")

In [ ]:
# Train single-layer NN (Adam) with MSE loss
print("\n" + "="*60)
print("Test 5: Single-Layer NN (Adam, MSE Loss)")
print("="*60)

model_adam_mse = SingleLayerNN()
optimizer_adam_mse = torch.optim.Adam(model_adam_mse.parameters(), lr=0.001)
criterion_mse = nn.MSELoss()

scheduler_adam_mse = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_adam_mse,
    max_lr=0.01,
    epochs=500,
    steps_per_epoch=1,
    pct_start=0.3,
    anneal_strategy='cos'
)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_adam_mse = []
test_losses_adam_mse = []
best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
converged_epoch_adam_mse = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_adam_mse.train()
    optimizer_adam_mse.zero_grad()
    outputs = model_adam_mse(X_train_tensor)
    # Apply sigmoid to ensure [0,1] outputs, then compute MSE
    predictions = torch.sigmoid(outputs.squeeze())
    loss = criterion_mse(predictions, y_train_tensor)
    loss.backward()
    optimizer_adam_mse.step()
    scheduler_adam_mse.step()
    train_losses_adam_mse.append(loss.item())
    
    model_adam_mse.eval()
    with torch.no_grad():
        test_outputs = model_adam_mse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = criterion_mse(test_predictions, y_test_tensor)
        test_losses_adam_mse.append(test_loss.item())
    
    if test_loss.item() < best_loss - min_delta:
        best_loss = test_loss.item()
        best_model_state = model_adam_mse.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        current_lr = scheduler_adam_mse.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}, LR: {current_lr:.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_adam_mse = epoch + 1
        print(f"\nEarly stopping at epoch {converged_epoch_adam_mse}")
        model_adam_mse.load_state_dict(best_model_state)
        break

train_time_adam_mse = time.time() - start_time
print(f"\nTraining completed in {train_time_adam_mse:.2f} seconds")

model_adam_mse.eval()
with torch.no_grad():
    test_pred_adam_mse = torch.sigmoid(model_adam_mse(X_test_tensor)).numpy().flatten()

metrics_adam_mse = evaluate_model(y_test, test_pred_adam_mse, 'Single Layer NN (Adam, MSE)')
print(f"\nTest AUC: {metrics_adam_mse['auc']:.4f}")
print(f"Test AP: {metrics_adam_mse['average_precision']:.4f}")

In [ ]:
# Train single-layer NN (L-BFGS) with MSE loss
print("\n" + "="*60)
print("Test 5: Single-Layer NN (L-BFGS, MSE Loss)")
print("="*60)

model_lbfgs_mse = SingleLayerNN()
optimizer_lbfgs_mse = torch.optim.LBFGS(
    model_lbfgs_mse.parameters(),
    lr=0.01,
    max_iter=20
)
criterion_mse = nn.MSELoss()

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_lbfgs_mse = []
test_losses_lbfgs_mse = []
best_loss = float('inf')
epochs_no_improve = 0
converged_epoch_lbfgs_mse = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_lbfgs_mse.train()
    
    def closure():
        optimizer_lbfgs_mse.zero_grad()
        outputs = model_lbfgs_mse(X_train_tensor)
        predictions = torch.sigmoid(outputs.squeeze())
        loss = criterion_mse(predictions, y_train_tensor)
        loss.backward()
        return loss
    
    loss = optimizer_lbfgs_mse.step(closure)
    train_losses_lbfgs_mse.append(loss.item())
    
    model_lbfgs_mse.eval()
    with torch.no_grad():
        test_outputs = model_lbfgs_mse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = criterion_mse(test_predictions, y_test_tensor)
        test_losses_lbfgs_mse.append(test_loss.item())
    
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_lbfgs_mse = epoch + 1
        print(f"\nConverged at epoch {converged_epoch_lbfgs_mse}")
        break

train_time_lbfgs_mse = time.time() - start_time
print(f"\nTraining completed in {train_time_lbfgs_mse:.2f} seconds")

model_lbfgs_mse.eval()
with torch.no_grad():
    test_pred_lbfgs_mse = torch.sigmoid(model_lbfgs_mse(X_test_tensor)).numpy().flatten()

metrics_lbfgs_mse = evaluate_model(y_test, test_pred_lbfgs_mse, 'Single Layer NN (L-BFGS, MSE)')
print(f"\nTest AUC: {metrics_lbfgs_mse['auc']:.4f}")
print(f"Test AP: {metrics_lbfgs_mse['average_precision']:.4f}")

In [ ]:
# Train SimpleNN with MSE loss
print("\n" + "="*60)
print("Test 5: SimpleNN (MSE Loss)")
print("="*60)

simple_nn_mse = SimpleNN(
    input_dim=2,
    hidden_dims=(128, 64, 32),
    dropout_rate=0.3,
    use_class_weights=False  # No class weighting for MSE
)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)

criterion_simplenn_mse = nn.MSELoss()
optimizer_simplenn_mse = optim.AdamW(
    simple_nn_mse.parameters(),
    lr=0.001,
    weight_decay=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler_simplenn_mse = optim.lr_scheduler.OneCycleLR(
    optimizer_simplenn_mse,
    max_lr=0.01,
    epochs=60,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

num_epochs_simplenn = 60
patience_simplenn = 5
best_loss_simplenn = float('inf')
epochs_no_improve_simplenn = 0
best_model_state_simplenn = None

train_losses_simplenn_mse = []
test_losses_simplenn_mse = []

start_time = time.time()

for epoch in range(num_epochs_simplenn):
    simple_nn_mse.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer_simplenn_mse.zero_grad()
        outputs = simple_nn_mse(batch_X)
        predictions = torch.sigmoid(outputs.squeeze())
        loss = criterion_simplenn_mse(predictions, batch_y)
        loss.backward()
        optimizer_simplenn_mse.step()
        scheduler_simplenn_mse.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses_simplenn_mse.append(avg_train_loss)
    
    simple_nn_mse.eval()
    with torch.no_grad():
        test_outputs = simple_nn_mse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = criterion_simplenn_mse(test_predictions, y_test_tensor)
        test_losses_simplenn_mse.append(test_loss.item())
    
    if test_loss.item() < best_loss_simplenn:
        best_loss_simplenn = test_loss.item()
        best_model_state_simplenn = simple_nn_mse.state_dict().copy()
        epochs_no_improve_simplenn = 0
    else:
        epochs_no_improve_simplenn += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_simplenn} - "
              f"Train Loss: {avg_train_loss:.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve_simplenn >= patience_simplenn:
        print(f"\nEarly stopping at epoch {epoch+1}")
        simple_nn_mse.load_state_dict(best_model_state_simplenn)
        break

train_time_simplenn_mse = time.time() - start_time
print(f"\nTraining completed in {train_time_simplenn_mse:.2f} seconds")

simple_nn_mse.eval()
with torch.no_grad():
    test_pred_simplenn_mse = torch.sigmoid(simple_nn_mse(X_test_tensor)).numpy().flatten()

metrics_simplenn_mse = evaluate_model(y_test, test_pred_simplenn_mse, 'SimpleNN (MSE)')
print(f"\nTest AUC: {metrics_simplenn_mse['auc']:.4f}")
print(f"Test AP: {metrics_simplenn_mse['average_precision']:.4f}")

In [ ]:
# Train Ridge regression
print("\n" + "="*60)
print("Test 5: Ridge Regression (alpha=1.0)")
print("="*60)

ridge_mse = Ridge(alpha=1.0, random_state=42)

start_time = time.time()
ridge_mse.fit(X_train, y_train)
train_time_ridge_mse = time.time() - start_time

# Ridge.predict outputs raw values, clip to [0,1] for probability interpretation
test_pred_ridge_mse = np.clip(ridge_mse.predict(X_test), 0, 1)

metrics_ridge_mse = evaluate_model(y_test, test_pred_ridge_mse, 'Ridge (MSE)')

print(f"Training completed in {train_time_ridge_mse:.2f} seconds")
print(f"\nTest AUC: {metrics_ridge_mse['auc']:.4f}")
print(f"Test AP: {metrics_ridge_mse['average_precision']:.4f}")

In [ ]:
# Save Test 5 results

adam_mse_results = {
    'model': model_adam_mse,
    'test_pred': test_pred_adam_mse,
    'train_losses': train_losses_adam_mse,
    'test_losses': test_losses_adam_mse,
    'metrics': metrics_adam_mse,
    'train_time': train_time_adam_mse,
    'converged_epoch': converged_epoch_adam_mse
}
with open(f'{results_dir}/test5_single_layer_nn_adam_mse.pkl', 'wb') as f:
    pickle.dump(adam_mse_results, f)
print(f"Saved: {results_dir}/test5_single_layer_nn_adam_mse.pkl")

lbfgs_mse_results = {
    'model': model_lbfgs_mse,
    'test_pred': test_pred_lbfgs_mse,
    'train_losses': train_losses_lbfgs_mse,
    'test_losses': test_losses_lbfgs_mse,
    'metrics': metrics_lbfgs_mse,
    'train_time': train_time_lbfgs_mse,
    'converged_epoch': converged_epoch_lbfgs_mse
}
with open(f'{results_dir}/test5_single_layer_nn_lbfgs_mse.pkl', 'wb') as f:
    pickle.dump(lbfgs_mse_results, f)
print(f"Saved: {results_dir}/test5_single_layer_nn_lbfgs_mse.pkl")

simplenn_mse_results = {
    'model': simple_nn_mse,
    'test_pred': test_pred_simplenn_mse,
    'train_losses': train_losses_simplenn_mse,
    'test_losses': test_losses_simplenn_mse,
    'metrics': metrics_simplenn_mse,
    'train_time': train_time_simplenn_mse
}
with open(f'{results_dir}/test5_simple_nn_mse.pkl', 'wb') as f:
    pickle.dump(simplenn_mse_results, f)
print(f"Saved: {results_dir}/test5_simple_nn_mse.pkl")

ridge_mse_results = {
    'model': ridge_mse,
    'test_pred': test_pred_ridge_mse,
    'metrics': metrics_ridge_mse,
    'train_time': train_time_ridge_mse
}
with open(f'{results_dir}/test5_ridge.pkl', 'wb') as f:
    pickle.dump(ridge_mse_results, f)
print(f"Saved: {results_dir}/test5_ridge.pkl")

print("\nTest 5 results saved successfully!")

## Test 6: Weighted MSE Loss

This test applies sample weighting to MSE loss to handle class imbalance:
1. Same models as Test 5 but with weighted MSE
2. Positive samples weighted by neg/pos ratio
3. Should combine benefits of MSE (calibration) with class balance handling

Expected: Similar or better calibration than Test 5, better handling of minority class

In [ ]:
# Calculate sample weights for weighted MSE
sample_weight_ratio = num_negatives / num_positives
print(f"\nSample weight ratio for Test 6: {sample_weight_ratio:.2f}")
print(f"  Positive examples will be weighted {sample_weight_ratio:.2f}x")
print(f"  Negative examples will be weighted 1.0x")

In [ ]:
# Train single-layer NN (Adam) with weighted MSE loss
print("\n" + "="*60)
print("Test 6: Single-Layer NN (Adam, Weighted MSE Loss)")
print("="*60)

model_adam_wmse = SingleLayerNN()
optimizer_adam_wmse = torch.optim.Adam(model_adam_wmse.parameters(), lr=0.001)

scheduler_adam_wmse = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_adam_wmse,
    max_lr=0.01,
    epochs=500,
    steps_per_epoch=1,
    pct_start=0.3,
    anneal_strategy='cos'
)

# Create sample weights tensor
sample_weights = torch.where(y_train_tensor == 1, sample_weight_ratio, 1.0)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_adam_wmse = []
test_losses_adam_wmse = []
best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
converged_epoch_adam_wmse = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_adam_wmse.train()
    optimizer_adam_wmse.zero_grad()
    outputs = model_adam_wmse(X_train_tensor)
    predictions = torch.sigmoid(outputs.squeeze())
    # Weighted MSE
    loss = (sample_weights * (predictions - y_train_tensor)**2).mean()
    loss.backward()
    optimizer_adam_wmse.step()
    scheduler_adam_wmse.step()
    train_losses_adam_wmse.append(loss.item())
    
    model_adam_wmse.eval()
    with torch.no_grad():
        test_outputs = model_adam_wmse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = ((test_predictions - y_test_tensor)**2).mean()  # Unweighted for validation
        test_losses_adam_wmse.append(test_loss.item())
    
    if test_loss.item() < best_loss - min_delta:
        best_loss = test_loss.item()
        best_model_state = model_adam_wmse.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        current_lr = scheduler_adam_wmse.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}, LR: {current_lr:.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_adam_wmse = epoch + 1
        print(f"\nEarly stopping at epoch {converged_epoch_adam_wmse}")
        model_adam_wmse.load_state_dict(best_model_state)
        break

train_time_adam_wmse = time.time() - start_time
print(f"\nTraining completed in {train_time_adam_wmse:.2f} seconds")

model_adam_wmse.eval()
with torch.no_grad():
    test_pred_adam_wmse = torch.sigmoid(model_adam_wmse(X_test_tensor)).numpy().flatten()

metrics_adam_wmse = evaluate_model(y_test, test_pred_adam_wmse, 'Single Layer NN (Adam, Weighted MSE)')
print(f"\nTest AUC: {metrics_adam_wmse['auc']:.4f}")
print(f"Test AP: {metrics_adam_wmse['average_precision']:.4f}")

In [ ]:
# Train single-layer NN (L-BFGS) with weighted MSE loss
print("\n" + "="*60)
print("Test 6: Single-Layer NN (L-BFGS, Weighted MSE Loss)")
print("="*60)

model_lbfgs_wmse = SingleLayerNN()
optimizer_lbfgs_wmse = torch.optim.LBFGS(
    model_lbfgs_wmse.parameters(),
    lr=0.01,
    max_iter=20
)

max_epochs = 500
patience = 50
min_delta = 1e-6

train_losses_lbfgs_wmse = []
test_losses_lbfgs_wmse = []
best_loss = float('inf')
epochs_no_improve = 0
converged_epoch_lbfgs_wmse = max_epochs

start_time = time.time()

for epoch in range(max_epochs):
    model_lbfgs_wmse.train()
    
    def closure():
        optimizer_lbfgs_wmse.zero_grad()
        outputs = model_lbfgs_wmse(X_train_tensor)
        predictions = torch.sigmoid(outputs.squeeze())
        loss = (sample_weights * (predictions - y_train_tensor)**2).mean()
        loss.backward()
        return loss
    
    loss = optimizer_lbfgs_wmse.step(closure)
    train_losses_lbfgs_wmse.append(loss.item())
    
    model_lbfgs_wmse.eval()
    with torch.no_grad():
        test_outputs = model_lbfgs_wmse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = ((test_predictions - y_test_tensor)**2).mean()
        test_losses_lbfgs_wmse.append(test_loss.item())
    
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{max_epochs} - "
              f"Train Loss: {loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve >= patience:
        converged_epoch_lbfgs_wmse = epoch + 1
        print(f"\nConverged at epoch {converged_epoch_lbfgs_wmse}")
        break

train_time_lbfgs_wmse = time.time() - start_time
print(f"\nTraining completed in {train_time_lbfgs_wmse:.2f} seconds")

model_lbfgs_wmse.eval()
with torch.no_grad():
    test_pred_lbfgs_wmse = torch.sigmoid(model_lbfgs_wmse(X_test_tensor)).numpy().flatten()

metrics_lbfgs_wmse = evaluate_model(y_test, test_pred_lbfgs_wmse, 'Single Layer NN (L-BFGS, Weighted MSE)')
print(f"\nTest AUC: {metrics_lbfgs_wmse['auc']:.4f}")
print(f"Test AP: {metrics_lbfgs_wmse['average_precision']:.4f}")

In [ ]:
# Train SimpleNN with weighted MSE loss
print("\n" + "="*60)
print("Test 6: SimpleNN (Weighted MSE Loss)")
print("="*60)

simple_nn_wmse = SimpleNN(
    input_dim=2,
    hidden_dims=(128, 64, 32),
    dropout_rate=0.3,
    use_class_weights=False
)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)

optimizer_simplenn_wmse = optim.AdamW(
    simple_nn_wmse.parameters(),
    lr=0.001,
    weight_decay=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler_simplenn_wmse = optim.lr_scheduler.OneCycleLR(
    optimizer_simplenn_wmse,
    max_lr=0.01,
    epochs=60,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

num_epochs_simplenn = 60
patience_simplenn = 5
best_loss_simplenn = float('inf')
epochs_no_improve_simplenn = 0
best_model_state_simplenn = None

train_losses_simplenn_wmse = []
test_losses_simplenn_wmse = []

start_time = time.time()

for epoch in range(num_epochs_simplenn):
    simple_nn_wmse.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer_simplenn_wmse.zero_grad()
        outputs = simple_nn_wmse(batch_X)
        predictions = torch.sigmoid(outputs.squeeze())
        # Get weights for this batch
        batch_weights = torch.where(batch_y == 1, sample_weight_ratio, 1.0)
        loss = (batch_weights * (predictions - batch_y)**2).mean()
        loss.backward()
        optimizer_simplenn_wmse.step()
        scheduler_simplenn_wmse.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses_simplenn_wmse.append(avg_train_loss)
    
    simple_nn_wmse.eval()
    with torch.no_grad():
        test_outputs = simple_nn_wmse(X_test_tensor)
        test_predictions = torch.sigmoid(test_outputs.squeeze())
        test_loss = ((test_predictions - y_test_tensor)**2).mean()
        test_losses_simplenn_wmse.append(test_loss.item())
    
    if test_loss.item() < best_loss_simplenn:
        best_loss_simplenn = test_loss.item()
        best_model_state_simplenn = simple_nn_wmse.state_dict().copy()
        epochs_no_improve_simplenn = 0
    else:
        epochs_no_improve_simplenn += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_simplenn} - "
              f"Train Loss: {avg_train_loss:.6f}, Test Loss: {test_loss.item():.6f}")
    
    if epochs_no_improve_simplenn >= patience_simplenn:
        print(f"\nEarly stopping at epoch {epoch+1}")
        simple_nn_wmse.load_state_dict(best_model_state_simplenn)
        break

train_time_simplenn_wmse = time.time() - start_time
print(f"\nTraining completed in {train_time_simplenn_wmse:.2f} seconds")

simple_nn_wmse.eval()
with torch.no_grad():
    test_pred_simplenn_wmse = torch.sigmoid(simple_nn_wmse(X_test_tensor)).numpy().flatten()

metrics_simplenn_wmse = evaluate_model(y_test, test_pred_simplenn_wmse, 'SimpleNN (Weighted MSE)')
print(f"\nTest AUC: {metrics_simplenn_wmse['auc']:.4f}")
print(f"Test AP: {metrics_simplenn_wmse['average_precision']:.4f}")

In [ ]:
# Train Ridge regression with sample weights
print("\n" + "="*60)
print("Test 6: Ridge Regression (Weighted)")
print("="*60)

ridge_wmse = Ridge(alpha=1.0, random_state=42)

# Create sample weights for Ridge
sample_weights_np = np.where(y_train == 1, sample_weight_ratio, 1.0)

start_time = time.time()
ridge_wmse.fit(X_train, y_train, sample_weight=sample_weights_np)
train_time_ridge_wmse = time.time() - start_time

test_pred_ridge_wmse = np.clip(ridge_wmse.predict(X_test), 0, 1)

metrics_ridge_wmse = evaluate_model(y_test, test_pred_ridge_wmse, 'Ridge (Weighted MSE)')

print(f"Training completed in {train_time_ridge_wmse:.2f} seconds")
print(f"\nTest AUC: {metrics_ridge_wmse['auc']:.4f}")
print(f"Test AP: {metrics_ridge_wmse['average_precision']:.4f}")

In [ ]:
# Save Test 6 results

adam_wmse_results = {
    'model': model_adam_wmse,
    'test_pred': test_pred_adam_wmse,
    'train_losses': train_losses_adam_wmse,
    'test_losses': test_losses_adam_wmse,
    'metrics': metrics_adam_wmse,
    'train_time': train_time_adam_wmse,
    'converged_epoch': converged_epoch_adam_wmse
}
with open(f'{results_dir}/test6_single_layer_nn_adam_weighted_mse.pkl', 'wb') as f:
    pickle.dump(adam_wmse_results, f)
print(f"Saved: {results_dir}/test6_single_layer_nn_adam_weighted_mse.pkl")

lbfgs_wmse_results = {
    'model': model_lbfgs_wmse,
    'test_pred': test_pred_lbfgs_wmse,
    'train_losses': train_losses_lbfgs_wmse,
    'test_losses': test_losses_lbfgs_wmse,
    'metrics': metrics_lbfgs_wmse,
    'train_time': train_time_lbfgs_wmse,
    'converged_epoch': converged_epoch_lbfgs_wmse
}
with open(f'{results_dir}/test6_single_layer_nn_lbfgs_weighted_mse.pkl', 'wb') as f:
    pickle.dump(lbfgs_wmse_results, f)
print(f"Saved: {results_dir}/test6_single_layer_nn_lbfgs_weighted_mse.pkl")

simplenn_wmse_results = {
    'model': simple_nn_wmse,
    'test_pred': test_pred_simplenn_wmse,
    'train_losses': train_losses_simplenn_wmse,
    'test_losses': test_losses_simplenn_wmse,
    'metrics': metrics_simplenn_wmse,
    'train_time': train_time_simplenn_wmse
}
with open(f'{results_dir}/test6_simple_nn_weighted_mse.pkl', 'wb') as f:
    pickle.dump(simplenn_wmse_results, f)
print(f"Saved: {results_dir}/test6_simple_nn_weighted_mse.pkl")

ridge_wmse_results = {
    'model': ridge_wmse,
    'test_pred': test_pred_ridge_wmse,
    'metrics': metrics_ridge_wmse,
    'train_time': train_time_ridge_wmse
}
with open(f'{results_dir}/test6_ridge_weighted.pkl', 'wb') as f:
    pickle.dump(ridge_wmse_results, f)
print(f"Saved: {results_dir}/test6_ridge_weighted.pkl")

print("\nTest 6 results saved successfully!")
print("\nAll tests complete (Tests 1-6)!")
print("  Test 1-2: Baseline (BCEWithLogitsLoss, no class weighting)")
print("  Test 3: Added class weighting (BCEWithLogitsLoss)")
print("  Test 4: Added StandardScaler (BCEWithLogitsLoss, class weighting)")
print("  Test 5: MSE Loss (no class weighting, better calibration)")
print("  Test 6: Weighted MSE Loss (class balance + calibration)")